In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

from src.etl.loader import load_all_data

data      = load_all_data()
pl        = data['profitandloss']
bs        = data['balancesheet']
companies = data['companies']

print("Data loaded!")
print(f"P&L rows: {len(pl)}")
print(f"BS rows:  {len(bs)}")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Data loaded!
P&L rows: 1164
BS rows:  1165


In [2]:
# We need both P&L and Balance Sheet columns together
# to compute ROE and ROCE
merged = pd.merge(
    pl,
    bs[['company_id', 'year', 'equity_capital', 'reserves',
        'borrowings', 'total_assets', 'fixed_assets', 'depreciation'
        if 'depreciation' in bs.columns else 'fixed_assets']],
    on=['company_id', 'year'],
    how='inner'
)

print(f"Merged rows: {len(merged)}")
print(f"Columns: {merged.columns.tolist()}")

Merged rows: 1082
Columns: ['id', 'company_id', 'year', 'sales', 'expenses', 'operating_profit', 'opm_percentage', 'other_income', 'interest', 'depreciation', 'profit_before_tax', 'tax_percentage', 'net_profit', 'eps', 'dividend_payout', 'equity_capital', 'reserves', 'borrowings', 'total_assets', 'fixed_assets', 'fixed_assets']


In [3]:
def compute_npm(df):
    """
    Net Profit Margin = Net Profit / Sales × 100
    Returns None if Sales is zero or missing.
    """
    df = df.copy()
    df['net_profit_margin_pct'] = np.where(
        df['sales'] > 0,
        (df['net_profit'] / df['sales'] * 100).round(2),
        None
    )
    return df

merged = compute_npm(merged)

print("Net Profit Margin — Sample:")
print(merged[['company_id', 'year', 'sales',
              'net_profit', 'net_profit_margin_pct']].head(10))

Net Profit Margin — Sample:
  company_id     year  sales  net_profit net_profit_margin_pct
0        ABB  2012-12   1653         145                  8.77
1        ABB  2014-03   2276         198                   8.7
2        ABB  2015-03   2289         229                  10.0
3        ABB  2016-03   2614         255                  9.76
4        ABB  2017-03   2903         277                  9.54
5        ABB  2018-03   3298         401                 12.16
6        ABB  2019-03   3679         450                 12.23
7        ABB  2020-03   4093         593                 14.49
8        ABB  2021-03   4310         691                 16.03
9        ABB  2022-03   4913         799                 16.26


In [4]:
def compute_opm(df):
    """
    Operating Profit Margin = Operating Profit / Sales × 100
    Returns None if Sales is zero or missing.
    """
    df = df.copy()
    df['operating_profit_margin_pct'] = np.where(
        df['sales'] > 0,
        (df['operating_profit'] / df['sales'] * 100).round(2),
        None
    )
    return df

merged = compute_opm(merged)

print("Operating Profit Margin — Sample:")
print(merged[['company_id', 'year',
              'operating_profit_margin_pct']].head(10))

Operating Profit Margin — Sample:
  company_id     year operating_profit_margin_pct
0        ABB  2012-12                       12.22
1        ABB  2014-03                       11.73
2        ABB  2015-03                       13.63
3        ABB  2016-03                       13.96
4        ABB  2017-03                       13.71
5        ABB  2018-03                       15.92
6        ABB  2019-03                       16.44
7        ABB  2020-03                       18.49
8        ABB  2021-03                       21.39
9        ABB  2022-03                       22.02


In [5]:
def compute_roe(df):
    """
    ROE = Net Profit / (Equity Capital + Reserves) × 100
    Returns None if total equity is zero or negative.
    """
    df = df.copy()
    df['total_equity'] = df['equity_capital'] + df['reserves']

    df['return_on_equity_pct'] = np.where(
        df['total_equity'] > 0,
        (df['net_profit'] / df['total_equity'] * 100).round(2),
        None
    )
    return df

merged = compute_roe(merged)

print("Return on Equity — Sample:")
print(merged[['company_id', 'year',
              'total_equity', 'return_on_equity_pct']].head(10))

# Quick sanity check — TCS ROE should be high
tcs_roe = merged[merged['company_id'] == 'TCS'][
    ['year', 'return_on_equity_pct']
].tail(5)
print("\nTCS ROE (last 5 years):")
print(tcs_roe)

Return on Equity — Sample:
  company_id     year  total_equity return_on_equity_pct
0        ABB  2012-12         647.0                22.41
1        ABB  2014-03         788.0                25.13
2        ABB  2015-03         937.0                24.44
3        ABB  2016-03        1195.0                21.34
4        ABB  2017-03        1387.0                19.97
5        ABB  2018-03        1693.0                23.69
6        ABB  2019-03        2008.0                22.41
7        ABB  2020-03        2431.0                24.39
8        ABB  2021-03        2602.0                26.56
9        ABB  2022-03        2820.0                28.33

TCS ROE (last 5 years):
        year return_on_equity_pct
981  2020-03                38.57
982  2021-03                37.67
983  2022-03                43.13
984  2023-03                46.78
985  2024-03                50.94


In [6]:
def compute_roce(df):
    """
    ROCE = EBIT / (Equity + Reserves + Borrowings) × 100
    EBIT = Operating Profit - Depreciation
    Returns None if capital employed is zero or negative.
    """
    df = df.copy()

    # EBIT = Operating Profit - Depreciation
    # Use depreciation from P&L if available
    df['ebit'] = df['operating_profit'] - df['depreciation']

    # Capital Employed = Equity + Borrowings
    df['capital_employed'] = df['total_equity'] + df['borrowings']

    df['return_on_capital_pct'] = np.where(
        df['capital_employed'] > 0,
        (df['ebit'] / df['capital_employed'] * 100).round(2),
        None
    )
    return df

merged = compute_roce(merged)

print("Return on Capital Employed — Sample:")
print(merged[['company_id', 'year',
              'ebit', 'capital_employed',
              'return_on_capital_pct']].head(10))

Return on Capital Employed — Sample:
  company_id     year    ebit  capital_employed return_on_capital_pct
0        ABB  2012-12   183.0             647.0                 28.28
1        ABB  2014-03   245.0             788.0                 31.09
2        ABB  2015-03   297.0             937.0                  31.7
3        ABB  2016-03   351.0            1195.0                 29.37
4        ABB  2017-03   382.0            1387.0                 27.54
5        ABB  2018-03   509.0            1693.0                 30.06
6        ABB  2019-03   588.0            2008.0                 29.28
7        ABB  2020-03   697.0            2606.0                 26.75
8        ABB  2021-03   864.0            2755.0                 31.36
9        ABB  2022-03  1016.0            2972.0                 34.19


In [7]:
# Check how many valid values we computed
print("Profitability Ratios — Coverage Summary:")
print(f"  Total rows: {len(merged)}")
print(f"  NPM computed:  {merged['net_profit_margin_pct'].notna().sum()}")
print(f"  OPM computed:  {merged['operating_profit_margin_pct'].notna().sum()}")
print(f"  ROE computed:  {merged['return_on_equity_pct'].notna().sum()}")
print(f"  ROCE computed: {merged['return_on_capital_pct'].notna().sum()}")

print("\nSample — TCS latest year:")
tcs_latest = merged[merged['company_id'] == 'TCS'].tail(1)[
    ['company_id', 'year', 'net_profit_margin_pct',
     'operating_profit_margin_pct', 'return_on_equity_pct',
     'return_on_capital_pct']
]
print(tcs_latest.to_string(index=False))

Profitability Ratios — Coverage Summary:
  Total rows: 1082
  NPM computed:  1081
  OPM computed:  1069
  ROE computed:  1082
  ROCE computed: 1070

Sample — TCS latest year:
company_id    year net_profit_margin_pct operating_profit_margin_pct return_on_equity_pct return_on_capital_pct
       TCS 2024-03                 19.14                       26.69                50.94                 60.21
